# Physics — Solar PV I–V and P–V performance analysis
## FDP Day 5 — AI-Powered Scientific Research with Google Colab

**Objective:** Plot photovoltaic I–V and P–V curves and compare maximum power points across irradiance and module temperature.

**Matching dataset:** `09_Physics_Solar_PV_IV_Characteristics.csv`

All bundled CSVs are **synthetic teaching datasets** with plausible ranges and units, not real experimental observations. Replace them with approved research data for actual studies. Avoid uploading identifiable patient or confidential data.

**Workflow:** Upload → inspect → analyse → visualise → interpret → export.


## 1. Setup and upload the matching CSV

In [ ]:
!pip -q install seaborn scikit-learn opencv-python-headless
import io, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from IPython.display import display
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid",context="notebook")
plt.rcParams["figure.figsize"]=(9,5)
uploaded=files.upload()
csv_name=next(f for f in uploaded if f.lower().endswith(".csv"))
df=pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("Loaded",csv_name,"shape:",df.shape)
display(df.head()); display(df.describe(include="all").T)


## 2. Upload image input
**Suggested image:** `09_sample_thermal_false_colour.png`. Supported formats: JPG, JPEG, PNG. Use original-resolution, focused images with consistent lighting/magnification. The supplied image is an illustrative teaching sample.

In [ ]:
from PIL import Image
image_upload=files.upload()
image_name=next(f for f in image_upload if f.lower().endswith((".jpg",".jpeg",".png")))
img=np.array(Image.open(io.BytesIO(image_upload[image_name])).convert("RGB"))
print("Image shape:",img.shape)
plt.figure(figsize=(10,6));plt.imshow(img);plt.axis("off");plt.title("Uploaded image");plt.show()


## 2. Solar photovoltaic I–V and P–V curves

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,5))
for irr,g in df.groupby("irradiance_W_m2"):
    g=g.sort_values("voltage_V")
    ax[0].plot(g.voltage_V,g.current_A,"o-",label=f"{irr} W/m²")
    ax[1].plot(g.voltage_V,g.power_W,"o-",label=f"{irr} W/m²")
ax[0].set(xlabel="Voltage (V)",ylabel="Current (A)",title="I–V curves")
ax[1].set(xlabel="Voltage (V)",ylabel="Power (W)",title="P–V curves")
for a in ax:a.legend()
plt.tight_layout();plt.show()
idx=df.groupby(["irradiance_W_m2","module_temperature_C"]).power_W.idxmax()
mpp=df.loc[idx,["irradiance_W_m2","module_temperature_C","voltage_V","current_A","power_W"]]
display(mpp.sort_values(["irradiance_W_m2","module_temperature_C"]).round(3))


## Research interpretation checklist
- State the research question, data source, sample size and units.
- Explain two findings supported by the charts.
- Identify uncertainty, confounders, measurement limitations and possible bias.
- State what additional data or validation is required before publication.
- Do not claim causation from correlation or clinical/biological diagnosis from simple image segmentation.


In [ ]:
# Export a copy of the data with any new analysis columns
df.to_csv("processed_research_data.csv",index=False)
files.download("processed_research_data.csv")


## 3. Optional physics extension: thermal-image intensity mapping

Use the uploaded thermal-camera export (JPG/PNG) to create a false-colour display, compare two regions of interest and plot a pixel-intensity profile. **Unless the file contains calibrated radiometric temperature values, these are relative image intensities, not temperatures in °C.**

In [ ]:
import cv2
gray=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)
false_colour=cv2.applyColorMap(gray,cv2.COLORMAP_INFERNO)
false_colour=cv2.cvtColor(false_colour,cv2.COLOR_BGR2RGB)
H,W=gray.shape
fig,ax=plt.subplots(1,2,figsize=(13,5))
ax[0].imshow(img);ax[0].set_title("Uploaded thermal export")
ax[1].imshow(false_colour);ax[1].set_title("False-colour intensity map")
for a in ax:a.axis("off")
plt.tight_layout();plt.show()
row=H//2
plt.plot(gray[row,:]);plt.title("Horizontal pixel-intensity profile");plt.xlabel("Pixel position");plt.ylabel("Grayscale intensity (0–255)");plt.show()
# Edit ROI coordinates to compare two regions
roi1=gray[int(H*.2):int(H*.5),int(W*.15):int(W*.4)]
roi2=gray[int(H*.2):int(H*.5),int(W*.6):int(W*.85)]
print("ROI 1 mean intensity:",round(float(roi1.mean()),2),"ROI 2 mean intensity:",round(float(roi2.mean()),2))
print("Intensity difference only; not a calibrated temperature difference.")
